## memmap

In [3]:
import numpy as np
import os
import torch

file_name = "test.dat"

arr = np.memmap(file_name, dtype="float32", mode="w+", shape=(10000, 1000))

arr[:] = np.random.random((10000, 1000)).astype('float32')

arr.flush()
del arr

arr = np.memmap(file_name, dtype="float32", mode="r", shape=(10000, 1000))
print(arr[0, :5])

x_t = torch.from_numpy(arr)
print("number of element: ", x_t.numel())
del x_t
del arr

os.remove(file_name)

[0.24415357 0.9897821  0.6963122  0.45736313 0.04637669]
number of element:  10000000


## fancy indexing

In [6]:
x = np.arange(1, 11)
print(x)

indices = [1, 4, 9]
select_eles = x[indices]
print(select_eles)

indices_2d = np.array([[0, 4], [3, 7]])
print(x[indices_2d])

x[indices_2d] = 99
print(x)

[ 1  2  3  4  5  6  7  8  9 10]
[ 2  5 10]
[[1 5]
 [4 8]]
[99  2  3 99 99  6  7 99  9 10]


In [9]:
x[0:4]

array([99,  2,  3, 99])

In [8]:
X = np.arange(12).reshape((3, 4))
print(X)

# Select elements (0, 2), (1, 1), and (2, 3)
row_indices = np.array([0, 1, 2])
col_indices = np.array([2, 1, 3])
selected_elements = X[row_indices, col_indices]
print(selected_elements)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[ 2  5 11]


## as_strided

In [7]:
import numpy.lib.stride_tricks  as stride_tricks

X = np.asarray(range(1, 26), np.int8).reshape(5, 5)

# 输出数组 A 的-1维度: 每行提取3个元素
# 输出数组 A 的-2维度: 每列提取3个元素
shape = (3, 3)  # 输出数组 A 的 shape

# int8 每个元素占1字节
# 输出数组 A 的-1维度: 相邻元素在原数组 X 中间隔5个元素 (eg. 1 → 6, 7 → 12, 3 → 8), 即 1x5=5 字节
# 输出数组 A 的-2维度: 相邻元素 (跨行) 在原数组 X 中间隔1个元素 (eg. 1 → 2, 2 → 3, 11 → 12), 即 1x1=1 字节
strides = (1, 5)

A = stride_tricks.as_strided(X, shape=shape, strides=strides)

# 等价于 x[:3,:3].T

print(X)
print(A)

[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]
 [16 17 18 19 20]
 [21 22 23 24 25]]
[[ 1  6 11]
 [ 2  7 12]
 [ 3  8 13]]


In [9]:
X = np.asarray(range(1,26), np.int16).reshape(5,5)

# 输出数组 A 的-1维度: 输出数组内层有3列 (eg. [1 2 3], [3 4 5], [11 12 13], [18 19 20])
# 输出数组 A 的-2维度: 输出数组内层有3行 (eg. [1 6 11], [3 8 13], [12 17 22])
# 输出数组 A 的-3维度: 输出数组中间层有2块 (eg. 第一块: [[1 2 3], [6 7 8], [11 12 13]]; 第二块: [[3 4 5], [8 9 10], [13 14 15]])
# 输出数组 A 的-4维度: 输出数组外层有2块
shape = (2, 2, 3, 3)  # 输出数组 A 的 shape

# int16 每个元素占2字节
# 输出数组 A 的-1维度: 相邻元素 (跨列) 在原数组 X 中间隔1个元素 (eg. 1 → 2, 17 → 18, 24 → 25), 即 1x2=2 字节
# 输出数组 A 的-2维度: 相邻元素 (跨行) 在原数组 X 中间隔5个元素 (eg. 1 → 6, 2 → 7, 3 → 8), 即 5x2=10 字节
# 输出数组 A 的-3维度: 相邻元素 (跨块) 在原数组 X 中间隔2个元素 (eg. 1 → 3, 6 → 8, 21 → 23, 23 → 25), 即 2x2=4 字节
# 输出数组 A 的-4维度: 相邻元素 (跨块) 在原数组 X 中间隔10个元素 (eg. 1 → 11, 2 → 12, 15 → 25), 即 10x2=20 字节
strides = (20, 4, 10, 2)

A = stride_tricks.as_strided(X, shape=shape, strides=strides)

print(X)
print(A)

[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]
 [16 17 18 19 20]
 [21 22 23 24 25]]
[[[[ 1  2  3]
   [ 6  7  8]
   [11 12 13]]

  [[ 3  4  5]
   [ 8  9 10]
   [13 14 15]]]


 [[[11 12 13]
   [16 17 18]
   [21 22 23]]

  [[13 14 15]
   [18 19 20]
   [23 24 25]]]]


## 长序列中随机采样 Batch 数据

In [11]:
import numpy as np

dataset_size = 1000
batch_size = 4
context_length = 5
seq_len = context_length + 1

start_indices = np.random.randint(low=0, high=dataset_size - seq_len, size=(batch_size,))

print(f"起始索引形状：{start_indices.shape}")
print(f"起始索引内容：{start_indices}")

起始索引形状：(4,)
起始索引内容：[333 435 468 341]


In [12]:
offsets = np.arange(seq_len)

# 3. 利用广播机制构建索引矩阵
# start_indices[:, np.newaxis] 将形状从 (B,) 变为 (B, 1)
# (B, 1) + (seq_len,) -> 广播为 (B, seq_len)
index_matrix = start_indices[:, np.newaxis] + offsets

print(f"偏移量形状：{offsets.shape}")
print(f"索引矩阵形状：{index_matrix.shape}")
print(f"索引矩阵内容:\n{index_matrix}")

偏移量形状：(6,)
索引矩阵形状：(4, 6)
索引矩阵内容:
[[333 334 335 336 337 338]
 [435 436 437 438 439 440]
 [468 469 470 471 472 473]
 [341 342 343 344 345 346]]


In [13]:
dataset = np.arange(1000, 1000 + dataset_size)

batch_data = dataset[index_matrix]

print(f"最终数据块形状：{batch_data.shape}")
print(f"最终数据块内容:\n{batch_data}")

最终数据块形状：(4, 6)
最终数据块内容:
[[1333 1334 1335 1336 1337 1338]
 [1435 1436 1437 1438 1439 1440]
 [1468 1469 1470 1471 1472 1473]
 [1341 1342 1343 1344 1345 1346]]


### 等效实现: stack

In [15]:
batch_data = np.stack([ dataset[start:start+seq_len] for start in start_indices])
print(batch_data)

[[1333 1334 1335 1336 1337 1338]
 [1435 1436 1437 1438 1439 1440]
 [1468 1469 1470 1471 1472 1473]
 [1341 1342 1343 1344 1345 1346]]
